# 投资 Dashboard - 持仓可视化

**功能**: 股价走势 + 技术因子 + 信号展示

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots

db_path = Path('../../data/invest.db')
conn = sqlite3.connect(db_path)

stock_names = {
    '002648.SZ': '卫星化学',
    '002594.SZ': '比亚迪',
    '000833.SZ': '粤桂股份',
    '300442.SZ': '润泽科技',
    '516120.SH': '化工 50ETF'
}

print("✅ 数据加载完成")

In [ ]:
selected_code = '002594.SZ'
selected_name = stock_names[selected_code]
print(f"分析股票：{selected_name} ({selected_code})")

daily_df = pd.read_sql_query(f"SELECT * FROM stock_daily WHERE ts_code='{selected_code}' ORDER BY trade_date", conn)
daily_df['trade_date'] = pd.to_datetime(daily_df['trade_date'], format='%Y%m%d')

tech_df = pd.read_sql_query(f"SELECT * FROM calc_technical_factors WHERE ts_code='{selected_code}' ORDER BY trade_date", conn)
tech_df['trade_date'] = pd.to_datetime(tech_df['trade_date'], format='%Y%m%d')

print(f"数据量：{len(daily_df)} 条")

In [ ]:
fig1 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.7, 0.3])
fig1.add_trace(go.Scatter(x=daily_df['trade_date'], y=daily_df['close'], name='收盘价', line=dict(color='blue', width=2)), row=1, col=1)
fig1.add_trace(go.Scatter(x=tech_df['trade_date'], y=tech_df['boll_position'], name='布林带位置', line=dict(color='purple', width=2)), row=2, col=1)
fig1.add_hline(y=0.8, line_dash="dash", line_color="red", row=2, col=1, annotation_text="超买")
fig1.add_hline(y=0.2, line_dash="dash", line_color="green", row=2, col=1, annotation_text="超卖")
fig1.update_layout(title=f'{selected_name} - 股价 + 布林带', height=600, hovermode='x unified')
fig1.show()

In [ ]:
fig2 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.5, 0.5])
fig2.add_trace(go.Scatter(x=tech_df['trade_date'], y=tech_df['macd_dif_pct'], name='MACD 分位数', line=dict(color='blue', width=2)), row=1, col=1)
fig2.add_hline(y=0.8, line_dash="dash", line_color="red", row=1, col=1)
fig2.add_hline(y=0.2, line_dash="dash", line_color="green", row=1, col=1)
fig2.add_trace(go.Scatter(x=tech_df['trade_date'], y=tech_df['rsi_pct'], name='RSI 分位数', line=dict(color='orange', width=2)), row=2, col=1)
fig2.add_hline(y=0.8, line_dash="dash", line_color="red", row=2, col=1)
fig2.add_hline(y=0.2, line_dash="dash", line_color="green", row=2, col=1)
fig2.update_layout(title=f'{selected_name} - MACD + RSI', height=600, hovermode='x unified')
fig2.show()

In [ ]:
confidence_df = pd.read_sql_query("SELECT * FROM calc_confidence", conn)
confidence_df['name'] = confidence_df['ts_code'].map(stock_names)

fig3 = go.Figure()
fig3.add_trace(go.Bar(x=confidence_df['name'], y=confidence_df['confidence'] * 100,
    marker_color=['green' if s == 'strong_buy' else 'yellow' for s in confidence_df['signal_type']],
    text=[f'{c:.1f}%' for c in confidence_df['confidence'] * 100], textposition='outside'))
fig3.update_layout(title='持仓标的置信度对比', xaxis_title='股票', yaxis_title='置信度 (%)', yaxis=dict(range=[0, 100]), height=500)
fig3.show()

fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=confidence_df['fundamental_score'], y=confidence_df['technical_score'],
    mode='markers+text', text=confidence_df['name'], textposition='top center',
    marker=dict(size=15, color=confidence_df['confidence'], colorscale='RdYlGn', showscale=True)))
fig4.update_layout(title='持仓分析：技术面 vs 基本面', xaxis_title='基本面评分', yaxis_title='技术面评分',
    xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1]), height=600)
fig4.add_hline(y=0.5, line_dash="dash", line_color="gray")
fig4.add_vline(x=0.5, line_dash="dash", line_color="gray")
fig4.show()

conn.close()
print("✅ Dashboard 完成")